# Prepare training data

Load **mixed_v2** and **sandwitch_v2** `.pt` chunks, merge, shuffle, and split into **train / validation / test** for word-level classification.

Each sample:
- **x** (`text`): full mixed document string
- **y** (`label`): list of `0` (human) / `1` (AI) per word, aligned with `text.split()`

In [7]:
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import Dataset
import pickle

from generator.load_pt import list_chunks, load_chunk

PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "generator"))

# --- paths (chunk dirs from 1_data_generate.py) ---
MIXED_V2_DIR = PROJECT_DIR / "output" / "mixed_v2"
SANDWITCH_V2_DIR = PROJECT_DIR / "output" / "sandwitch_v2"
# Subfolders to scan under each mode dir (None = train, validation, test if present)
SOURCE_SPLITS = None  # e.g. ["train"] to only load train chunks

# --- train / val / test split ---
TRAIN_RATIO = 0.8
VAL_RATIO = 0.05
TEST_RATIO = 0.15
RANDOM_SEED = 42

# --- output ---
PREPARED_DIR = PROJECT_DIR / "output" / "prepared"

In [ ]:
def labels_to_list(labels) -> list[int]:
    if isinstance(labels, torch.Tensor):
        return [int(x) for x in labels.tolist()]
    return [int(x) for x in labels]


def validate_sample(text: str, label: list[int]) -> None:
    words = text.split()
    if len(words) != len(label):
        raise ValueError(
            f"Word count ({len(words)}) != label count ({len(label)}); sample misaligned."
        )
    if label and any(x not in (0, 1) for x in label):
        raise ValueError("Labels must be 0 (human) or 1 (AI).")


def discover_source_splits(output_dir: Path, source_splits=None) -> list[str]:
    if source_splits is not None:
        return list(source_splits)
    if not output_dir.exists():
        return []
    return sorted(
        d.name
        for d in output_dir.iterdir()
        if d.is_dir() and any(d.glob("*.pt"))
    )


def load_samples_from_dir(
    output_dir: Path, source_mode: str, source_splits=None
) -> list[dict]:
    """Load all samples from every .pt chunk under output_dir/<split>/."""
    if not output_dir.exists():
        print(f"Warning: missing directory {output_dir}")
        return []

    splits = discover_source_splits(output_dir, source_splits)
    if not splits:
        print(f"Warning: no .pt chunks under {output_dir}")
        return []

    samples = []
    n_chunks = 0
    for source_split in splits:
        paths = list_chunks(output_dir, source_split)
        n_chunks += len(paths)
        for path in paths:
            data = load_chunk(path)
            mode = data.get("generation_mode", source_mode)
            for i in range(len(data["texts"])):
                text = data["texts"][i]
                label = labels_to_list(data["labels"][i])
                try:
                    validate_sample(text, label)
                except ValueError:
                    print(f"Error: {path} {i}")
                    continue
                samples.append(
                    {
                        "text": text,
                        "label": label,
                        "source_mode": mode,
                        "minipile_index": int(data["indices"][i]),
                        "model": data["models"][i],
                        "chunk": path.name,
                    }
                )
    print(
        f"{source_mode}: {len(samples)} samples from {n_chunks} chunks "
        f"({', '.join(splits)}) in {output_dir}"
    )
    return samples


def merge_sources(*sample_lists: list[dict]) -> list[dict]:
    merged = []
    for batch in sample_lists:
        merged.extend(batch)
    return merged


def shuffle_samples(samples: list[dict], seed: int) -> list[dict]:
    rng = random.Random(seed)
    out = samples.copy()
    rng.shuffle(out)
    return out


def split_train_val_test(
    samples: list[dict],
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
) -> tuple[list[dict], list[dict], list[dict]]:
    total_ratio = train_ratio + val_ratio + test_ratio
    if abs(total_ratio - 1.0) > 1e-6:
        raise ValueError(f"Ratios must sum to 1.0, got {total_ratio}")

    n = len(samples)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train = samples[:n_train]
    val = samples[n_train : n_train + n_val]
    test = samples[n_train + n_val :]
    return train, val, test

In [9]:
import os
# create directory if it doesn't exist
os.makedirs("pickles", exist_ok=True)

# load all mixed samples and save as pickle
mixed_samples = load_samples_from_dir(MIXED_V2_DIR, "mixed_v2", SOURCE_SPLITS)
sandwich_samples = load_samples_from_dir(SANDWITCH_V2_DIR, "sandwitch_v2", SOURCE_SPLITS)
print("Mixed samples length: ", len(mixed_samples))
print("Sandwich samples length: ", len(sandwich_samples))

with open("pickles/mixed_samples.pkl", "wb") as f:
    pickle.dump([{"text": d["text"], "label": d["label"]} for d in mixed_samples], f)

with open("pickles/sandwich_samples.pkl", "wb") as f:
    pickle.dump([{"text": d["text"], "label": d["label"]} for d in sandwich_samples], f)

mixed_v2: 29941 samples from 31 chunks (test, train) in /home/ine/projects/human-ai-mixed-data-generator/output/mixed_v2
sandwitch_v2: 13478 samples from 15 chunks (test, train, validation) in /home/ine/projects/human-ai-mixed-data-generator/output/sandwitch_v2
Mixed samples length:  29941
Sandwich samples length:  13478


In [ ]:
# load human data from pile dataset (from dataset/minipile/data)
from random import shuffle
import pandas as pd

def env_value(key, default=""):
    value = os.environ.get(key, default)
    return value.split("#", 1)[0].strip()

DATA_DIR = Path(env_value("MINIPILE_DATA_DIR", "dataset/minipile/data"))

files = sorted(Path(DATA_DIR).glob(f"train-*.parquet"))
last_file = files[-1]

table = pd.read_parquet(last_file, columns=["text"])
content = table["text"].tolist()

random.seed(42)
shuffle(content)

content = content[:30000]

with open("pickles/human_samples.pkl", "wb") as f:
    pickle.dump([{"text": d, "label": [0] * len(d.split())} for d in content], f)


In [18]:
# load ai data from ai-text-detection-training

AI_DATA_DIR = Path("dataset/ai-text-detection-training")

# pyarrow does not expand wildcard strings here; collect files first.
ai_files = sorted(AI_DATA_DIR.glob("*.parquet"))
if not ai_files:
    raise FileNotFoundError(
        f"No parquet files found in {AI_DATA_DIR}. "
        "Check path and extracted dataset files."
    )

# read all parquet files as one dataset/table
df = pd.read_parquet(ai_files)
ai_texts = df[df["label"] == 1]["text"].tolist()

print(f"Loaded {len(ai_texts)} AI texts from {len(ai_files)} parquet files")
print(ai_texts[0])

# save ai texts to pickle
with open("pickles/ai_texts.pkl", "wb") as f:
    pickle.dump([{"text": d, "label": [1] * len(d.split())} for d in ai_texts], f)


Loaded 36741 AI texts from 2 parquet files
Seven years after OWL was ratified as a W3C recommendation and two years after the release of the OWL 2 recommendation, its adoption on the Web remains uneven. While certain constructs such as owl:sameAs enjoy widespread use, many other OWL features are scarcely employed by Linked Data publishers. This pattern suggests that, despite the availability of straightforward implementations and the tractable profiles introduced in OWL 2, the Linked Data community has not yet settled on a definitive standard fragment. In this study we (1) assess the current prevalence of OWL across the Web of Data, (2) identify the subset of OWL that is effectively used and practical in this environment—concluding that it corresponds to a simplified profile derived from OWL RL, and (3) propose and evaluate a new profile, dubbed OWL LD (Linked Data).


In [19]:
# load all pickles
with open("pickles/mixed_samples.pkl", "rb") as f:
    mixed_samples = pickle.load(f)

with open("pickles/sandwich_samples.pkl", "rb") as f:
    sandwich_samples = pickle.load(f)

with open("pickles/human_samples.pkl", "rb") as f:
    human_samples = pickle.load(f)

with open("pickles/ai_texts.pkl", "rb") as f:
    ai_samples = pickle.load(f)

print("Mixed samples: ", len(mixed_samples))
print("Sandwich samples: ", len(sandwich_samples))
print("Human samples: ", len(human_samples))
print("AI samples: ", len(ai_samples))


Mixed samples:  29941
Sandwich samples:  13478
Human samples:  30000
AI samples:  36741


In [59]:
# merge all samples
all_samples = merge_sources(mixed_samples, sandwich_samples, human_samples, ai_samples)

if not all_samples:
    raise RuntimeError(
        "No samples found. Generate data first, e.g.\n"
        "  python 1_data_generate.py --mode mixed_v2 ...\n"
        "  python 1_data_generate.py --mode sandwitch_v2 ..."
    )

all_samples = shuffle_samples(all_samples, RANDOM_SEED)
train, val, test = split_train_val_test(all_samples, TRAIN_RATIO, VAL_RATIO, TEST_RATIO)

print(f"Total merged: {len(all_samples)}")
print(f"train={len(train)}, val={len(val)}, test={len(test)}")

# save each split
for name, split_samples in [("train", train), ("validation", val), ("test", test)]:
    with open(f"pickles/{name}_samples.pkl", "wb") as f:
        pickle.dump(split_samples, f)

Total merged: 110160
train=88128, val=5508, test=16524
